<a href="https://colab.research.google.com/github/DamjiStratton/ed_workforce_navigator/blob/main/occupation-based-index-for-workforce-ai-navigator%20(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **OBI-WAN**: **O**ccupation **B**ased **I**ndex for **W**orkforce **A**I **N**avigator

# 1. Notebook Roadmap

This notebook is organized as follows:

1. **Setup**: Install and import libraries, configure BigQuery and Gemini.
2. **Data Access**: Connect to BigQuery and load job and embedding tables.
3. **Agent Tools**: Define custom BigQuery tools for skills, career forecasting, and institution lookup.
4. **Agent Definition**: Create the OBI-WAN LLM agent with instructions and tool wiring.
5. **Deployment (ADK Web UI)**: Launch the agent using ADK Web Runner and open the web UI.
6. **Technical Concepts Demonstrated**: Summarize how this project uses ADK concepts.
7. **How to Run / Reproduce**: Simple steps for running this notebook.
8. **Video Demonstration**: Link to YouTube overview and demo.

blablabla

# 2. Setup: Libraries, Credentials, and Clients

In [ ]:
!pip install google-cloud-bigquery

import os
import json
from typing import Optional

import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient

# BigQuery / GCP
from google.cloud import bigquery
from google.oauth2 import service_account

# Gemini / Generative AI
import google.generativeai as genai

# ADK
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.tool_context import ToolContext
from google.adk.runners import InMemoryRunner   # optional – keep if you use it
from google.adk.sessions import InMemorySessionService  # optional

# For ADK Web UI proxy in Kaggle
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers


In [ ]:
# --- CONFIGURATION ---
GCP_PROJECT_ID = "dashboard-458701"
REGION = "us-west1"

# Table IDs
JOB_DATA_TABLE_ID = "onet.job_search_table"
JOB_EMBED_TABLE_ID = "onet.job_embeddings"
INST_DATA_TABLE_ID = "ipeds.institution_search_table"

# AI Score Data
AI_SCORE_TABLE_ID = "onet.final_ai_applicability_multiplicative"
AI_SCORE_COL = "AI_Applicability_Score"

# Embedding model constant
EMBED_MODEL = "models/text-embedding-004"

# --- INITIALIZATION ---
def initialize_services():
    """
    Initialize Gemini and BigQuery using Kaggle Secrets.
    Returns:
        bq_client: an authenticated BigQuery client.
    """
    try:
        user_secrets = UserSecretsClient()

        # 1. Setup Gemini
        google_api_key = user_secrets.get_secret("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = google_api_key
        genai.configure(api_key=google_api_key)

        # 2. Setup BigQuery
        gcp_key_json = user_secrets.get_secret("GCP_KEY")
        credentials = service_account.Credentials.from_service_account_info(json.loads(gcp_key_json))
        bq_client = bigquery.Client(
            project=GCP_PROJECT_ID,
            credentials=credentials,
            location=REGION,
        )

        print("✅ BigQuery and Gemini initialized")
        return bq_client

    except Exception as e:
        print(f"❌ Initialization Error: {e}")
        return None

# Create global BigQuery client
bq_client = initialize_services()


# 3. Data sanity check

In [ ]:
sql = """
SELECT DISTINCT soc_code, Title, Description
FROM `dashboard-458701.onet.job_search_table`
WHERE Title IS NOT NULL
ORDER BY soc_code
LIMIT 5
"""
jobs_df = bq_client.query(sql).to_dataframe()
jobs_df.head()


sql = """
SELECT soc_code, Title, full_text, embedding
FROM `dashboard-458701.onet.job_embeddings`
LIMIT 5
"""
emb_df = bq_client.query(sql).to_dataframe()
emb_df.head()

# 4. Agent Tools: Skills, Jobs, and Institutions

In [ ]:
# --- GLOBAL CACHES ---
job_registry_df = None
job_emb_matrix = None


# --- Registry Loader ---
def load_job_registry():
    global job_registry_df, job_emb_matrix
    if job_registry_df is None:
        print("⏳ Loading OBI-WAN Job Registry...")
        sql = f"""
            SELECT e.soc_code, e.Title, e.embedding, s.{AI_SCORE_COL} as ai_score
            FROM `{GCP_PROJECT_ID}.{JOB_EMBED_TABLE_ID}` e
            LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s
                ON e.soc_code = s.soc_code
            WHERE e.embedding IS NOT NULL
        """
        try:
            job_registry_df = bq_client.query(sql).to_dataframe()
            job_registry_df['ai_score'] = pd.to_numeric(job_registry_df['ai_score'], errors='coerce').fillna(0.5)
            job_emb_matrix = np.array(job_registry_df["embedding"].tolist())
            print(f"✅ OBI-WAN Registry Loaded: {len(job_registry_df)} jobs.")
        except Exception as e:
            print(f"❌ Error loading registry: {e}")


# --- HELPER FUNCTIONS ---
def get_ai_narrative(score):
    score = float(score)
    HIGH_THRESHOLD = 0.5
    LOW_THRESHOLD = 0.2

    if score >= HIGH_THRESHOLD:
        return (f"⚠️ **High AI Integration Zone (Score: {score:.2f})**")
    elif score <= LOW_THRESHOLD:
        return (f"🛡️ **Human-Centric Zone (Score: {score:.2f})**")
    else:
        return (f"⚖️ **Hybrid Zone (Score: {score:.2f})**")

def normalize_award_level(user_text: Optional[str]) -> Optional[str]:
    if not user_text: return None
    t = user_text.lower()
    if "bachelor" in t or "undergrad" in t: return "bachelor"
    if "master" in t or "graduate" in t and "doctor" not in t: return "master"
    if "post-master" in t or "post master" in t: return "post-master"
    if "postbacc" in t or "post-bacc" in t: return "postbacc"
    if "associate" in t or "assoc" in t: return "associate"
    if "certificate" in t or "cert" in t: return "certificate"
    return None

def normalize_modality(user_text: Optional[str]) -> Optional[str]:
    if not user_text: return None
    t = user_text.lower()
    if "online" in t or "remote" in t: return "DE ENTIRELY"
    if "hybrid" in t or "mixed" in t: return "DE SOME"
    if "campus" in t or "person" in t or "f2f" in t: return "F2F"
    return None


# --- TOOL 1: Get Job Requirements
def get_job_requirements(
    tool_context: ToolContext,
    job_title: str,
    category: str = "Skill"   # default so tool-calling works
) -> str:
    """
    Finds skills, knowledge, or abilities for a job title.
    Args:
        job_title: The job (e.g., "Translator").
        category: (optional) 'Skill', 'Knowledge', or 'Ability'.
                  Defaults to 'Skill' if not provided.
    """
    # 1. Make sure registry is loaded (embeddings + AI score)
    load_job_registry()
    if job_registry_df is None or job_registry_df.empty:
        return "Error: job registry is empty or could not be loaded."

    # 2. Semantic match job title → SOC code
    try:
        resp = genai.embed_content(model=EMBED_MODEL, content=job_title)
        q_vec = np.array(resp["embedding"])

        sims = np.dot(job_emb_matrix, q_vec) / (
            np.linalg.norm(job_emb_matrix, axis=1) * np.linalg.norm(q_vec) + 1e-8
        )
        best_idx = int(np.argmax(sims))
        best_row = job_registry_df.iloc[best_idx]

        best_title = best_row["Title"]
        best_soc = best_row["soc_code"]
        ai_score = best_row["ai_score"]
    except Exception as e:
        return f"Navigation Error: Could not locate job '{job_title}'. Details: {e}"

    # 3. Save best match for follow-ups
    tool_context.state["last_job_title"] = best_title

    # 4. Normalize category → DB values
    cat_lower = category.lower()
    if "knowledge" in cat_lower:
        db_category = "Knowledge"
    elif "ability" in cat_lower or "abilities" in cat_lower:
        db_category = "Ability"
    else:
        db_category = "Skill"

    # 5. Use parameterized query (old working pattern)
    job_config_items = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("category_param", "STRING", db_category),
            bigquery.ScalarQueryParameter("best_soc_code_param", "STRING", best_soc),
        ]
    )

    sql_get_items = f"""
        SELECT `Element Name` AS item_name
        FROM `{GCP_PROJECT_ID}.{JOB_DATA_TABLE_ID}`
        WHERE
            soc_code = @best_soc_code_param
            AND `Scale Name` = 'Importance'
            AND Category_ska = @category_param
        GROUP BY `Element Name`
        ORDER BY MAX(`Data Value`) DESC
        LIMIT 15
    """

    try:
        results = bq_client.query(sql_get_items, job_config=job_config_items).to_dataframe()
        if results.empty:
            skills_formatted = f"No specific {db_category} data found."
        else:
            # bullet list formatting
            skills_formatted = "\n".join(
                f"- {row['item_name']}" for _, row in results.iterrows()
            )
    except Exception as e:
        return f"Error retrieving skills from database: {e}"

    # 6. Combine with AI narrative
    return (
        f"### Analysis for {best_title}\n"
        f"{get_ai_narrative(ai_score)}\n\n"
        f"**Top {db_category}s:**\n{skills_formatted}"
    )


# TOOL 2: Get Jobs by Major
def get_jobs_by_major(tool_context: ToolContext, major_name: str) -> str:
    """
    Finds jobs for a college major and forecasts AI impact.
    Args:
        major_name: The major (e.g., "Psychology").
    """
    if not bq_client: return "Error: BigQuery not ready."
    tool_context.state['last_major_name'] = major_name

    sql = f"""
        SELECT DISTINCT j.Title, s.{AI_SCORE_COL} as score
        FROM `{GCP_PROJECT_ID}.{JOB_DATA_TABLE_ID}` j
        LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s ON j.soc_code = s.soc_code
        WHERE REGEXP_CONTAINS(j.CIPTitle, r'(?i){major_name}') AND j.Title IS NOT NULL LIMIT 50
    """
    try:
        df = bq_client.query(sql).to_dataframe()
        if df.empty: return f"No jobs found for major '{major_name}'."

        df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0.5)

        HIGH = 0.5
        LOW = 0.2

        high_ai = df[df['score'] >= HIGH]['Title'].head(5).tolist()
        low_ai = df[df['score'] <= LOW]['Title'].head(5).tolist()
        hybrid_ai = df[(df['score'] > LOW) & (df['score'] < HIGH)]['Title'].head(5).tolist()

        resp = f"### Career Forecast for {major_name}\n"
        if high_ai: resp += f"**🤖 Augmentation Path (High AI):**\n" + ", ".join(high_ai) + "\n\n"
        if low_ai: resp += f"**🛡️ Safe Haven (Human-Centric):**\n" + ", ".join(low_ai) + "\n\n"
        if hybrid_ai: resp += f"**⚖️ Hybrid Roles:**\n" + ", ".join(hybrid_ai)
        return resp
    except Exception as e: return f"Error: {e}"



# TOOL 3: Get Institutions
def get_institutions_by_major(tool_context: ToolContext, major_name: Optional[str] = None, award_level: Optional[str] = None, modality: Optional[str] = None) -> str:
    """
    Finds top institutions for a major.
    Args:
        major_name: The major (e.g., "Psychology").
        award_level: "Bachelor", "Master", etc.
        modality: "Online", "On-Campus", etc.
    """
    if not bq_client: return "Error: BigQuery not initialized."

    if not major_name:
        major_name = tool_context.state.get('last_major_name')
        if not major_name: return "Please tell me which major you are interested in first."

    sql_base = f"FROM `{GCP_PROJECT_ID}.{INST_DATA_TABLE_ID}`"
    where_clauses = [f"REGEXP_CONTAINS(CIPTitle, r'(?i){major_name}')"]
    query_params = []

    award_keyword = normalize_award_level(award_level)
    if award_keyword:
        where_clauses.append("REGEXP_CONTAINS(LOWER(AWARD_LEVEL), @award_key)")
        query_params.append(bigquery.ScalarQueryParameter("award_key", "STRING", award_keyword))

    modality_code = normalize_modality(modality)
    if modality_code:
        where_clauses.append("MODALITY = @mod_code")
        query_params.append(bigquery.ScalarQueryParameter("mod_code", "STRING", modality_code))

    sql_query = f"""
        SELECT institution_name, SUM(CTOTALT) AS total_conferrals
        {sql_base}
        WHERE {" AND ".join(where_clauses)}
        GROUP BY institution_name
        HAVING total_conferrals > 0
        ORDER BY total_conferrals DESC LIMIT 10
    """

    try:
        job_config = bigquery.QueryJobConfig(query_parameters=query_params)
        results = bq_client.query(sql_query, job_config=job_config).to_dataframe()
        if results.empty: return f"No institutions found for '{major_name}' with those specific filters."

        inst_list = ", ".join([f"{i+1}. {row['institution_name']}" for i, row in results.iterrows()])
        return f"Top institutions for {major_name} ({award_level or 'All Levels'}, {modality or 'All Modalities'}): {inst_list}"
    except Exception as e: return f"Error: {e}"


# 5: Agent Definition

Agent Definition is:

# --- AGENT DEFINITION ---
root_agent = LlmAgent(
    name="OBI_WAN_Navigator",
    model=Gemini(model="models/gemini-2.5-flash"),
    
    instruction="""
    You are OBI-WAN (Occupation-Based Index for Workforce AI Navigator).
    
    **MANDATORY OUTPUT RULES:**
    1. **For Skills:** The tool will provide a list of 15 items. You MUST display the full list. **DO NOT SUMMARIZE.** Display them as a clean bulleted list.
    2. **For Jobs:** Group them by "High AI" (>0.5) and "Safe Haven" (<0.2).
    3. **For Institutions:** Ask for Degree Level and Modality before listing schools.
    
    **TONE:** Concise, Data-Driven, Strategic. Stop giving generic advice; focus on the data lists.
    """,
    tools=[get_job_requirements, get_jobs_by_major, get_institutions_by_major]
)

So all together,

In [ ]:
!mkdir career_coach_agent

**V3 Below is the Knowledge-Graph Based AND added program embedding data **

In [ ]:
%%writefile career_coach_agent/agent.py
import os
import json
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient

from google.cloud import bigquery
from google.oauth2 import service_account

import vertexai
from vertexai.language_models import TextEmbeddingModel, TextEmbeddingInput

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.tool_context import ToolContext


# ------------------------------------------------------------------
# --- CONFIGURATION ---
# ------------------------------------------------------------------
GCP_PROJECT_ID = "dashboard-458701"
REGION = "us-west1"

JOB_EMBED_TABLE_ID = "onet.job_embeddings_v2"
PROGRAM_EMBED_TABLE_ID = "ipeds.program_embeddings"

NODE_OCCUPATIONS = "onet.occupations_node"
NODE_COMPETENCIES = "onet.competencies_node"
NODE_PROGRAMS = "ipeds.programs_node"
NODE_INSTITUTIONS = "ipeds.institutions_node"

EDGE_JOB_COMPETENCY = "onet.job_requires_competencies_edge"
EDGE_PROGRAM_JOB = "crosswalk.program_leads_to_job_edge"
EDGE_INST_PROGRAM = "ipeds.institution_offers_programs_edge"

AI_SCORE_TABLE_ID = "onet.final_ai_applicability_multiplicative"
AI_SCORE_COL = "AI_Applicability_Score"

EMBED_MODEL_NAME = "text-embedding-004"


# ------------------------------------------------------------------
# --- INIT ---
# ------------------------------------------------------------------
def initialize_services():
    try:
        user_secrets = UserSecretsClient()

        google_api_key = user_secrets.get_secret("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = google_api_key

        gcp_key_json = user_secrets.get_secret("GCP_KEY")
        credentials = service_account.Credentials.from_service_account_info(
            json.loads(gcp_key_json)
        )

        bq = bigquery.Client(
            project=GCP_PROJECT_ID,
            credentials=credentials,
            location=REGION,
        )

        vertexai.init(project=GCP_PROJECT_ID, location=REGION, credentials=credentials)
        emb = TextEmbeddingModel.from_pretrained(EMBED_MODEL_NAME)

        print("✅ BigQuery + Vertex initialized")
        if os.getenv("OBIWAN_DEBUG", "0") == "1":
            print("🐞 OBIWAN_DEBUG=1")
        return bq, emb, google_api_key

    except Exception as e:
        print(f"❌ Initialization Error: {e}")
        return None, None, None


bq_client, embed_model, GOOGLE_API_KEY = initialize_services()


# ------------------------------------------------------------------
# --- CACHING GLOBALS ---
# ------------------------------------------------------------------
job_registry_df = None
job_emb_matrix = None

program_registry_df = None
program_emb_matrix = None


# ------------------------------------------------------------------
# 1) VECTOR SEARCH HELPERS
# ------------------------------------------------------------------
def load_job_registry():
    global job_registry_df, job_emb_matrix
    if not bq_client:
        raise RuntimeError("BigQuery client not initialized.")

    if job_registry_df is None:
        print("⏳ Loading Job Vector Registry...")
        sql = f"""
            SELECT soc_code AS onet_soc_code, title AS Title, embedding
            FROM `{GCP_PROJECT_ID}.{JOB_EMBED_TABLE_ID}`
            WHERE embedding IS NOT NULL
        """
        job_registry_df = bq_client.query(sql).to_dataframe()
        job_emb_matrix = np.array(job_registry_df["embedding"].tolist(), dtype=np.float32)

        if job_emb_matrix.size > 0:
            print(
                f"✅ Job Registry Loaded: {len(job_registry_df)} jobs (dim={job_emb_matrix.shape[1]})."
            )
        else:
            print("⚠️ Job Registry loaded, but embedding matrix is empty.")


def load_program_registry():
    global program_registry_df, program_emb_matrix
    if not bq_client:
        raise RuntimeError("BigQuery client not initialized.")

    if program_registry_df is None:
        print("⏳ Loading Program Vector Registry...")
        sql = f"""
            SELECT cip_code, title, embedding
            FROM `{GCP_PROJECT_ID}.{PROGRAM_EMBED_TABLE_ID}`
            WHERE embedding IS NOT NULL
        """
        program_registry_df = bq_client.query(sql).to_dataframe()
        program_emb_matrix = np.array(
            program_registry_df["embedding"].tolist(), dtype=np.float32
        )

        if program_emb_matrix.size > 0:
            print(
                f"✅ Program Registry Loaded: {len(program_registry_df)} programs (dim={program_emb_matrix.shape[1]})."
            )
        else:
            print("⚠️ Program Registry loaded, but embedding matrix is empty.")


def _embed_query_vertex(user_query: str) -> np.ndarray:
    if embed_model is None:
        raise RuntimeError("Embedding model not initialized.")
    if user_query is None or str(user_query).strip() == "":
        raise ValueError("User query is empty; cannot embed.")

    inputs = [TextEmbeddingInput(str(user_query), task_type="RETRIEVAL_QUERY")]
    result = embed_model.get_embeddings(inputs)
    return np.array(result[0].values, dtype=np.float32)


def _cosine_sims(matrix: np.ndarray, q_vec: np.ndarray) -> np.ndarray:
    denom = (np.linalg.norm(matrix, axis=1) * np.linalg.norm(q_vec) + 1e-8)
    return np.dot(matrix, q_vec) / denom


def find_topk_nodes_via_vector(
    user_query: str, type: str = "program", k: int = 3
) -> List[Dict[str, object]]:
    if type == "job":
        load_job_registry()
        df, matrix, id_col, title_col = (
            job_registry_df,
            job_emb_matrix,
            "onet_soc_code",
            "Title",
        )
    else:
        load_program_registry()
        df, matrix, id_col, title_col = (
            program_registry_df,
            program_emb_matrix,
            "cip_code",
            "title",
        )

    if df is None or df.empty or matrix is None or matrix.size == 0:
        return []

    q_vec = _embed_query_vertex(user_query)

    if matrix.shape[1] != q_vec.shape[0]:
        raise ValueError(
            f"Embedding dimension mismatch for type='{type}'. "
            f"Registry dim={matrix.shape[1]}, query dim={q_vec.shape[0]}."
        )

    sims = _cosine_sims(matrix, q_vec)
    k = max(1, int(k))
    k = min(k, len(sims))

    top_idx = np.argpartition(-sims, kth=k - 1)[:k]
    top_idx = top_idx[np.argsort(-sims[top_idx])]

    out: List[Dict[str, object]] = []
    for rank, idx in enumerate(top_idx, start=1):
        row = df.iloc[int(idx)]
        out.append(
            {
                "id": row[id_col],
                "title": row[title_col],
                "score": float(sims[int(idx)]),
                "rank": rank,
            }
        )

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print(f"---- OBIWAN DEBUG: find_topk_nodes_via_vector(type={type}) ----")
        print(f"Query: {user_query}")
        for m in out:
            print(
                f"  #{m['rank']}: {m['title']} (ID={m['id']}) score={m['score']:.4f}"
            )
        print("--------------------------------------------------------------")

    return out


def find_node_via_vector(user_query: str, type: str = "job") -> Tuple[Optional[str], Optional[str]]:
    matches = find_topk_nodes_via_vector(user_query, type=type, k=1)
    if not matches:
        return None, None
    return str(matches[0]["id"]), str(matches[0]["title"])


# ------------------------------------------------------------------
# 2) NORMALIZERS
# ------------------------------------------------------------------
def normalize_comp_category(category: Optional[str]) -> str:
    c = (category or "Skill").strip().lower()
    if "know" in c:
        return "Knowledge"
    if "abil" in c:
        return "Ability"
    return "Skill"


def normalize_award_level(user_text: Optional[str]) -> Optional[str]:
    if not user_text:
        return None
    t = user_text.strip().lower()
    if "associate" in t:
        return "associate"
    if "bachelor" in t or "undergrad degree" in t:
        return "bachelor"
    if "master" in t:
        return "master"
    if "doctor" in t or "phd" in t:
        return "doctor"
    if "graduate certificate" in t or "grad certificate" in t:
        return "grad_cert"
    if "undergraduate certificate" in t or "undergrad certificate" in t:
        return "ug_cert"
    if "certificate" in t:
        return "certificate"
    return None


def normalize_modality(user_text: Optional[str]) -> Optional[str]:
    if not user_text:
        return None
    t = user_text.strip().lower()
    if "online" in t or "remote" in t or "virtual" in t:
        return "DE ENTIRELY"
    if "hybrid" in t or "some" in t or "mixed" in t:
        return "DE SOME"
    if "on-campus" in t or "campus" in t or "person" in t or "f2f" in t:
        return "F2F"
    return None


def _award_sql_filters(aw_norm: Optional[str]) -> List[str]:
    if not aw_norm:
        return []
    if aw_norm == "associate":
        return ['e.award_level = "Associate\'s degree"']
    if aw_norm == "bachelor":
        return ['e.award_level = "Bachelor"']
    if aw_norm == "master":
        return ['e.award_level = "Master\'s degree"']
    if aw_norm == "doctor":
        return ["e.award_level LIKE 'Doctor%'", "e.award_level LIKE 'Doctoral%'"]
    if aw_norm == "ug_cert":
        return ["e.award_level LIKE 'Certificates of%'"]
    if aw_norm == "grad_cert":
        return [
            'e.award_level = "Post-master\'s certificate"',
            'e.award_level = "Postbaccalaureate certificate"',
        ]
    return []


# ------------------------------------------------------------------
# 3) TOOLS
# ------------------------------------------------------------------
def get_job_requirements(
    tool_context: ToolContext, job_title: str, category: str = "Skill", **kwargs
) -> str:
    if not bq_client:
        return "Error: BigQuery not ready."

    onet_code, official_title = find_node_via_vector(job_title, type="job")
    if not onet_code:
        return f"Could not find a match for '{job_title}'."

    tool_context.state["last_job_title"] = official_title
    ctype = normalize_comp_category(category)

    sql = f"""
        SELECT c.`Element Name` AS item_name, e.importance
        FROM `{GCP_PROJECT_ID}.{EDGE_JOB_COMPETENCY}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_COMPETENCIES}` c
          ON TRIM(e.element_id) = TRIM(c.`Element ID`)
        WHERE e.onet_soc_code = @soc
          AND c.type = @ctype
        QUALIFY ROW_NUMBER() OVER (PARTITION BY c.`Element Name` ORDER BY e.importance DESC) = 1
        ORDER BY e.importance DESC
        LIMIT 15
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("soc", "STRING", str(onet_code)),
            bigquery.ScalarQueryParameter("ctype", "STRING", ctype),
        ]
    )
    df = bq_client.query(sql, job_config=job_config).to_dataframe()
    if df.empty:
        return f"No {ctype} found for {official_title}."

    bullets = "\n".join(
        [f"- {r['item_name']} (Imp: {r['importance']})" for _, r in df.iterrows()]
    )
    return f"### {ctype} for {official_title}\n\n{bullets}"


def get_jobs_by_major(tool_context: ToolContext, major_name: str, **kwargs) -> str:
    if not bq_client:
        return "Error: BigQuery not ready."

    program_matches = find_topk_nodes_via_vector(major_name, type="program", k=3)
    if not program_matches:
        return f"Could not find a program match for '{major_name}'."

    tool_context.state["last_major_name"] = program_matches[0]["title"]
    tool_context.state["last_major_matches"] = program_matches

    cip_list = [str(m["id"]) for m in program_matches]
    match_lines = "\n".join([f"- #{m['rank']}: **{m['title']}**" for m in program_matches])

    try:
        sql = f"""
            WITH matched_programs AS (
              SELECT cip_code
              FROM UNNEST(@cip_codes) AS cip_code
            ),
            jobs AS (
              SELECT
                CAST(e.cip_code AS STRING) AS cip_code,
                CAST(e.onet_soc_code AS STRING) AS onet_soc_code,
                o.title AS job_title,
                COALESCE(CAST(s.{AI_SCORE_COL} AS FLOAT64), 0.5) AS ai_score
              FROM `{GCP_PROJECT_ID}.{EDGE_PROGRAM_JOB}` e
              JOIN matched_programs mp
                ON CAST(e.cip_code AS STRING) = mp.cip_code
              JOIN `{GCP_PROJECT_ID}.{NODE_OCCUPATIONS}` o
                ON CAST(e.onet_soc_code AS STRING) = CAST(o.onet_soc_code AS STRING)
              LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s
                ON CAST(e.onet_soc_code AS STRING) = CAST(s.soc_code AS STRING)
              WHERE o.title IS NOT NULL
            )
            SELECT
              cip_code,
              onet_soc_code,
              job_title,
              ai_score
            FROM jobs
            QUALIFY ROW_NUMBER() OVER (
                PARTITION BY cip_code, onet_soc_code, job_title
                ORDER BY ai_score DESC
            ) = 1
        """

        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ArrayQueryParameter("cip_codes", "STRING", cip_list)]
        )

        df = bq_client.query(sql, job_config=job_config).to_dataframe()
        if df.empty:
            return f"No direct career matches found for these program matches:\n{match_lines}"

        df["ai_score"] = pd.to_numeric(df["ai_score"], errors="coerce").fillna(0.5)

        if os.getenv("OBIWAN_DEBUG", "0") == "1":
            print("---- OBIWAN DEBUG: get_jobs_by_major ----")
            print(f"User query major: {major_name}")
            print("Top-3 program matches (debug):")
            for m in program_matches:
                print(f"  #{m['rank']}: {m['title']} (CIP={m['id']}) sim={m['score']:.4f}")
            print(f"Total rows returned: {len(df)}")

            if "onet_soc_code" in df.columns:
                socs = sorted(df["onet_soc_code"].astype(str).unique().tolist())
                print(f"Unique SOC count: {len(socs)}")
                print(f"SOC list: {socs}")

            if {"cip_code", "onet_soc_code"}.issubset(df.columns):
                per_cip = df.groupby("cip_code")["onet_soc_code"].nunique().sort_values(ascending=False)
                print("Unique SOC per CIP:")
                for cip, n in per_cip.items():
                    print(f"  CIP {cip}: {int(n)} SOCs")
            print("------------------------------------------")

        HIGH, LOW = 0.5, 0.2

        jobs_agg = (
            df.groupby("job_title", as_index=False)["ai_score"]
            .max()
            .sort_values("ai_score", ascending=False)
        )

        high_ai = jobs_agg[jobs_agg.ai_score > HIGH].job_title.head(10).tolist()
        safe = (
            jobs_agg[jobs_agg.ai_score < LOW]
            .sort_values("ai_score", ascending=True)
            .job_title.head(10)
            .tolist()
        )
        hybrid = jobs_agg[
            (jobs_agg.ai_score >= LOW) & (jobs_agg.ai_score <= HIGH)
        ].job_title.head(10).tolist()

        out = "### Program Matches (Top 3)\n\n" + match_lines + "\n\n"
        out += "### Career Options (combined from top 3 program matches)\n\n"
        out += "**🤖 High AI (>0.5):**\n" + ("\n".join(f"- {t}" for t in high_ai) or "- None") + "\n\n"
        out += "**🛡️ Safe Haven (<0.2):**\n" + ("\n".join(f"- {t}" for t in safe) or "- None") + "\n\n"
        out += "**⚖️ Hybrid (0.2–0.5):**\n" + ("\n".join(f"- {t}" for t in hybrid) or "- None")
        return out

    except Exception as e:
        return f"Tool error in get_jobs_by_major: {type(e).__name__}: {e}"


def _resolve_programs_from_job_via_reverse_edge(
    onet_soc_code: str,
    k: int = 3,
    award_level: Optional[str] = None,
    modality: Optional[str] = None,
    state: Optional[str] = None,
) -> List[Dict[str, object]]:
    """
    Reverse lookup: SOC -> CIPs using EDGE_PROGRAM_JOB, ranked by IPEDS completions (total_completers).
    Filters (award/modality/state) are applied on EDGE_INST_PROGRAM + institutions (NOT on EDGE_PROGRAM_JOB).
    """
    if not bq_client:
        return []

    mod_kw = normalize_modality(modality) if modality else None
    aw_norm = normalize_award_level(award_level) if award_level else None
    award_filters = _award_sql_filters(aw_norm)

    where2 = []
    params = [
        bigquery.ScalarQueryParameter("soc", "STRING", str(onet_soc_code)),
        bigquery.ScalarQueryParameter("k", "INT64", int(k)),
    ]

    if mod_kw:
        where2.append("e.modality = @mod")
        params.append(bigquery.ScalarQueryParameter("mod", "STRING", mod_kw))

    if award_filters:
        where2.append("(" + " OR ".join(award_filters) + ")")

    if state:
        where2.append("LOWER(n.state) = LOWER(@st)")
        params.append(bigquery.ScalarQueryParameter("st", "STRING", state))

    extra_where = (" AND " + " AND ".join(where2)) if where2 else ""

    sql = f"""
      WITH cips_for_soc AS (
        SELECT DISTINCT CAST(cip_code AS STRING) AS cip_code
        FROM `{GCP_PROJECT_ID}.{EDGE_PROGRAM_JOB}` x
        WHERE CAST(x.onet_soc_code AS STRING) = @soc
      ),
      cip_grads AS (
        SELECT
          c.cip_code,
          SUM(COALESCE(e.total_completers, 0)) AS grads
        FROM cips_for_soc c
        JOIN `{GCP_PROJECT_ID}.{EDGE_INST_PROGRAM}` e
          ON CAST(e.cip_code AS STRING) = c.cip_code
        JOIN `{GCP_PROJECT_ID}.{NODE_INSTITUTIONS}` n
          ON CAST(e.institution_id AS STRING) = CAST(n.institution_id AS STRING)
        WHERE 1=1 {extra_where}
        GROUP BY c.cip_code
      )
      SELECT
        g.cip_code,
        p.title AS program_title,
        g.grads
      FROM cip_grads g
      LEFT JOIN `{GCP_PROJECT_ID}.{NODE_PROGRAMS}` p
        ON CAST(p.cip_code AS STRING) = g.cip_code
      WHERE g.grads > 0
      ORDER BY g.grads DESC
      LIMIT @k
    """

    df = bq_client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).to_dataframe()

    out: List[Dict[str, object]] = []
    for i, r in df.iterrows():
        title = r["program_title"] if pd.notnull(r["program_title"]) else f"CIP {r['cip_code']}"
        out.append({"id": str(r["cip_code"]), "title": title, "rank": i + 1, "score": None})

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print("---- OBIWAN DEBUG: reverse SOC->CIP via completions ----")
        print(f"SOC: {onet_soc_code} | award_level={award_level} | modality={modality} | state={state}")
        for m in out:
            print(f"  #{m['rank']}: {m['title']} (CIP={m['id']})")
        print("--------------------------------------------------------")

    return out


def get_institutions_by_major(
    tool_context: ToolContext,
    major_name: Optional[str] = None,
    award_level: Optional[str] = None,
    modality: Optional[str] = None,
    state: Optional[str] = None,
    **kwargs
) -> str:
    if not bq_client:
        return "Error: BigQuery not initialized."

    resolved_major_name = major_name or tool_context.state.get("last_major_name")

    # Infer program from last job title if no major given
    if not resolved_major_name:
        last_job = tool_context.state.get("last_job_title")
        if last_job:
            soc, official_job = find_node_via_vector(last_job, type="job")

            inferred: List[Dict[str, object]] = []
            if soc:
                inferred = _resolve_programs_from_job_via_reverse_edge(
                    soc, k=3, award_level=award_level, modality=modality, state=state
                )

            # Fallback: vector-search program directly from job title
            if not inferred:
                inferred = find_topk_nodes_via_vector(official_job or last_job, type="program", k=3)

            if inferred:
                tool_context.state["last_major_name"] = inferred[0].get("title") or resolved_major_name
                tool_context.state["last_major_matches"] = inferred
                resolved_major_name = inferred[0].get("title") or resolved_major_name

    # Ask for missing fields ONCE (before any embedding call)
    missing = []
    if not resolved_major_name:
        missing.append("**major/program** (e.g., Counseling, Clinical Mental Health Counseling, Psychology)")
    if not award_level:
        missing.append("**degree level** (Associate, Bachelor, Master, Doctorate, Undergraduate Certificate, Graduate Certificate)")
    if not modality:
        missing.append("**modality** (Online, Hybrid, On-campus)")

    if missing:
        return (
            "To recommend institutions, I need: " + ", ".join(missing) +
            ".\n\nOptional: add a **state** (e.g., CA, WA) to filter results."
        )

    # Top-3 program candidates for CIP lookup
    top3 = find_topk_nodes_via_vector(resolved_major_name, type="program", k=3)

    if os.getenv("OBIWAN_DEBUG", "0") == "1":
        print("---- OBIWAN DEBUG: get_institutions_by_major ----")
        print(f"Resolved major: {resolved_major_name}")
        print("Top-3 program matches (CIP candidates):")
        for i, r in enumerate(top3, 1):
            title = r.get("title")
            cid = r.get("id")
            score = r.get("score")
            score_str = f"{float(score):.4f}" if score is not None else "None"
            print(f"  #{i}: {title} (CIP={cid}) sim={score_str}")
        print(f"Award level: {award_level}")
        print(f"Modality: {modality}")
        print(f"State: {state}")
        print("-----------------------------------------------")

    cip_code = top3[0].get("id") if top3 else None
    official_major = top3[0].get("title") if top3 and top3[0].get("title") else resolved_major_name

    # persist for downstream tools / follow-ups
    tool_context.state["last_major_name"] = official_major
    tool_context.state["last_major_matches"] = top3

    if not cip_code:
        return f"Could not find a program match for '{resolved_major_name}'."

    mod_kw = normalize_modality(modality)
    if not mod_kw:
        return "Modality not recognized. Try: **Online**, **Hybrid**, or **On-campus**."

    aw = normalize_award_level(award_level)
    if aw == "certificate":
        return (
            "When you say **certificate**, do you mean:\n"
            "- **Undergraduate Certificate**...\n"
            "- **Graduate Certificate**...\n"
            "- **Both**\n\n"
            "Reply with: **undergrad certificate**, **graduate certificate**, or **both**."
        )
    if not aw:
        return "Degree level not recognized."

    award_filters = _award_sql_filters(aw)

    where = ["e.modality = @mod"]
    params = [bigquery.ScalarQueryParameter("mod", "STRING", mod_kw)]

    if award_filters:
        where.append("(" + " OR ".join(award_filters) + ")")

    if state:
        where.append("LOWER(n.state) = LOWER(@st)")
        params.append(bigquery.ScalarQueryParameter("st", "STRING", state))

    sql = f"""
        SELECT
          n.name AS institution_name,
          n.state,
          e.award_level,
          e.modality,
          SUM(e.total_completers) AS grads
        FROM `{GCP_PROJECT_ID}.{EDGE_INST_PROGRAM}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_INSTITUTIONS}` n
          ON CAST(e.institution_id AS STRING) = CAST(n.institution_id AS STRING)
        WHERE CAST(e.cip_code AS STRING) = @cip
          AND {" AND ".join(where)}
        GROUP BY institution_name, state, award_level, modality
        HAVING grads > 0
        ORDER BY grads DESC
        LIMIT 10
    """

    job_config = bigquery.QueryJobConfig(
        query_parameters=params + [bigquery.ScalarQueryParameter("cip", "STRING", str(cip_code))]
    )

    df = bq_client.query(sql, job_config=job_config).to_dataframe()
    if df.empty:
        return (
            f"No institutions found for **{official_major}** with filters: {award_level} / {modality}"
            + (f" / {state}" if state else "")
        )

    out = (
        f"### Top Schools for {official_major}\n"
        f"Filters: **{award_level}**, **{modality}**" + (f", **{state}**" if state else "") + "\n\n"
    )
    for i, r in df.iterrows():
        out += f"{i+1}. **{r['institution_name']}** ({r['state']})\n"
    return out


# ------------------------------------------------------------------
# 4) ROOT AGENT
# ------------------------------------------------------------------
root_agent = LlmAgent(
    name="OBI_WAN_Navigator",
    model=Gemini(model="models/gemini-2.5-flash", api_key=GOOGLE_API_KEY),
    instruction="""
You are OBI-WAN (Occupation-Based Index for Workforce AI Navigator).

MANDATORY OUTPUT RULES:
1) For Skills: The tool will provide a list of 15 items. You MUST display the full list. DO NOT SUMMARIZE. Display them as a clean bulleted list.
2) For Jobs:
   - Show "Program Matches (Top 3)" as titles only (no CIP codes, no similarity scores).
   - Then group jobs by:
     - High AI (>0.5)
     - Safe Haven (<0.2)
     - Hybrid (0.2–0.5)
3) Institutions (critical):
   - If the user asks for schools and any of these are missing: major/program, degree level, modality,
     you MUST ask for ALL missing fields in ONE question.
   - If the user previously asked about skills for a specific job/occupation, you may infer the major/program automatically.
   - Use these user-friendly choices:
     - Modality: Online / Hybrid / On-campus
     - Degree level: Associate / Bachelor / Master / Doctorate / Undergraduate Certificate / Graduate Certificate
   - If user says only "certificate", you MUST ask: "Undergraduate Certificate, Graduate Certificate, or Both?"

TONE: Concise, Data-Driven, Strategic. Focus on the data lists.
""",
    tools=[get_job_requirements, get_jobs_by_major, get_institutions_by_major],
)


Writing career_coach_agent/agent.py


In [ ]:
import os, sys
os.environ["OBIWAN_DEBUG"] = "1"
sys.modules.pop("career_coach_agent.agent", None)
import career_coach_agent.agent as agent


✅ BigQuery + Vertex initialized
🐞 OBIWAN_DEBUG=1


/usr/local/lib/python3.11/dist-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


#### V2 Below is the Knowledge-Graph based Revised Script!

In [ ]:
%%writefile career_coach_agent/agent.py
import os
import json
import re
from typing import Optional, Tuple

import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient

from google.cloud import bigquery
from google.oauth2 import service_account
import google.generativeai as genai

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.tool_context import ToolContext

GCP_PROJECT_ID = "dashboard-458701"
REGION = "us-west1"

JOB_EMBED_TABLE_ID = "onet.job_embeddings"

NODE_OCCUPATIONS = "onet.occupations_node"
NODE_COMPETENCIES = "onet.competencies_node"
NODE_PROGRAMS = "ipeds.programs_node"
NODE_INSTITUTIONS = "ipeds.institutions_node"

EDGE_JOB_COMPETENCY = "onet.job_requires_competencies_edge"
EDGE_PROGRAM_JOB = "crosswalk.program_leads_to_job_edge"
EDGE_INST_PROGRAM = "ipeds.institution_offers_programs_edge"

AI_SCORE_TABLE_ID = "onet.final_ai_applicability_multiplicative"
AI_SCORE_COL = "AI_Applicability_Score"

EMBED_MODEL = "models/text-embedding-004"


def initialize_services():
    try:
        user_secrets = UserSecretsClient()

        google_api_key = user_secrets.get_secret("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = google_api_key
        genai.configure(api_key=google_api_key)

        gcp_key_json = user_secrets.get_secret("GCP_KEY")
        credentials = service_account.Credentials.from_service_account_info(json.loads(gcp_key_json))

        bq_client = bigquery.Client(
            project=GCP_PROJECT_ID,
            credentials=credentials,
            location=REGION,
        )
        print("✅ BigQuery and Gemini initialized")
        return bq_client
    except Exception as e:
        print(f"❌ Initialization Error: {e}")
        return None


bq_client = initialize_services()

job_registry_df = None
job_emb_matrix = None


def load_job_registry():
    global job_registry_df, job_emb_matrix
    if job_registry_df is None:
        print("⏳ Loading Vector Registry...")
        sql = f"""
            SELECT DISTINCT
                n.onet_soc_code,
                e.Title,
                e.embedding
            FROM `{GCP_PROJECT_ID}.{JOB_EMBED_TABLE_ID}` e
            JOIN `{GCP_PROJECT_ID}.{NODE_OCCUPATIONS}` n
              ON e.soc_code = n.onet_soc_code
            WHERE e.embedding IS NOT NULL
        """
        job_registry_df = bq_client.query(sql).to_dataframe()
        job_emb_matrix = np.array(job_registry_df["embedding"].tolist())
        print(f"✅ Vector Registry Loaded: {len(job_registry_df)} jobs.")


def find_node_via_vector(user_query: str) -> Tuple[Optional[str], Optional[str]]:
    load_job_registry()
    if job_registry_df is None or job_registry_df.empty:
        return None, None

    resp = genai.embed_content(model=EMBED_MODEL, content=user_query)
    q_vec = np.array(resp["embedding"])

    sims = np.dot(job_emb_matrix, q_vec) / (
        np.linalg.norm(job_emb_matrix, axis=1) * np.linalg.norm(q_vec) + 1e-8
    )
    best_idx = int(np.argmax(sims))
    best_row = job_registry_df.iloc[best_idx]

    print(
        f"🔎 Vector match → SOC: {best_row['onet_soc_code']} "
        f"(len={len(str(best_row['onet_soc_code']))}) | "
        f"Title: {best_row['Title']}"
    )
    return best_row["onet_soc_code"], best_row["Title"]


def normalize_comp_category(category: Optional[str]) -> str:
    c = (category or "Skill").strip().lower()
    if "know" in c:
        return "Knowledge"
    if "abil" in c:
        return "Ability"
    return "Skill"


def normalize_award_level(user_text: Optional[str]) -> Optional[str]:
    if not user_text:
        return None

    t = user_text.strip().lower()

    if "associate" in t:
        return "associate"

    if "bachelor" in t or "undergrad degree" in t:
        return "bachelor"

    if "master" in t:
        return "master"

    if "doctor" in t or "phd" in t:
        return "doctor"

    if "graduate certificate" in t or "grad certificate" in t:
        return "grad_cert"

    if "undergraduate certificate" in t or "undergrad certificate" in t:
        return "ug_cert"

    # fallback: generic "certificate"
    if "certificate" in t:
        return "certificate"

    return None




def normalize_modality(user_text: Optional[str]) -> Optional[str]:
    """
    Maps user-friendly modality wording to your edge table codes:
      Online -> DE ENTIRELY
      Hybrid -> DE SOME
      On-campus -> F2F
    """
    if not user_text:
        return None

    t = user_text.strip().lower()

    # online
    if "online" in t or "remote" in t or "distance" in t or "virtual" in t:
        return "DE ENTIRELY"

    # hybrid
    if "hybrid" in t or "some" in t or "mixed" in t:
        return "DE SOME"

    # on-campus / in-person
    if "on-campus" in t or "on campus" in t or "campus" in t or "in person" in t or "in-person" in t or "f2f" in t:
        return "F2F"

    # allow direct codes too (power users)
    if t in {"de entirely", "de some", "f2f"}:
        return t.upper() if t == "f2f" else t.upper().replace(" ", "_")

    return None




def get_job_requirements(
    tool_context: ToolContext,
    job_title: str,
    category: str = "Skill",
    **kwargs
) -> str:
    onet_code, official_title = find_node_via_vector(job_title)
    if not onet_code:
        return f"Could not find a match for '{job_title}'."

    tool_context.state["last_job_title"] = official_title
    ctype = normalize_comp_category(category)

    sql = f"""
        SELECT
          c.`Element Name` AS item_name,
          e.importance
        FROM `{GCP_PROJECT_ID}.{EDGE_JOB_COMPETENCY}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_COMPETENCIES}` c
          ON TRIM(e.element_id) = TRIM(c.`Element ID`)
        WHERE e.onet_soc_code = '{onet_code}'
          AND c.type = '{ctype}'
        QUALIFY ROW_NUMBER() OVER (
          PARTITION BY c.`Element Name`
          ORDER BY e.importance DESC
        ) = 1
        ORDER BY e.importance DESC
        LIMIT 15
    """
    df = bq_client.query(sql).to_dataframe()
    if df.empty:
        return f"No {ctype} found for {official_title}."

    bullets = "\n".join([f"- {r['item_name']} (Imp: {r['importance']})" for _, r in df.iterrows()])
    return f"### {ctype} for {official_title}\n\n{bullets}"



def get_jobs_by_major(tool_context: ToolContext, major_name: str, **kwargs) -> str:
    if not bq_client:
        return "Error: BigQuery not ready."

    tool_context.state["last_major_name"] = major_name

    try:
        # safer regex handling
        major_pattern = "(?i)" + re.escape((major_name or "").strip())

        sql = f"""
            WITH matching_cips AS (
              SELECT DISTINCT CAST(cip_code AS STRING) AS cip_code
              FROM `{GCP_PROJECT_ID}.ipeds.programs_node`
              WHERE REGEXP_CONTAINS(title, @major)
            ),
            jobs AS (
              SELECT
                o.title AS job_title,
                COALESCE(s.{AI_SCORE_COL}, 0.5) AS ai_score
              FROM `{GCP_PROJECT_ID}.crosswalk.program_leads_to_job_edge` e
              JOIN matching_cips c
                ON CAST(e.cip_code AS STRING) = c.cip_code
              JOIN `{GCP_PROJECT_ID}.onet.occupations_node` o
                ON CAST(e.onet_soc_code AS STRING) = CAST(o.onet_soc_code AS STRING)
              LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s
                ON CAST(e.onet_soc_code AS STRING) = CAST(s.soc_code AS STRING)
              WHERE o.title IS NOT NULL
            )
            SELECT job_title, ai_score
            FROM jobs
            QUALIFY ROW_NUMBER() OVER (
              PARTITION BY job_title
              ORDER BY ai_score DESC
            ) = 1
        """

        cfg = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("major", "STRING", major_pattern)]
        )

        df = bq_client.query(sql, job_config=cfg).to_dataframe()
        if df.empty:
            return f"No jobs found for major '{major_name}'."

        df["ai_score"] = pd.to_numeric(df["ai_score"], errors="coerce").fillna(0.5)

        HIGH, LOW = 0.5, 0.2

        high_ai = (
            df[df.ai_score > HIGH]
            .sort_values("ai_score", ascending=False)
            .job_title.head(10).tolist()
        )
        safe = (
            df[df.ai_score < LOW]
            .sort_values("ai_score", ascending=True)
            .job_title.head(10).tolist()
        )
        hybrid = (
            df[(df.ai_score >= LOW) & (df.ai_score <= HIGH)]
            .sort_values("ai_score", ascending=False)
            .job_title.head(10).tolist()
        )

        out = f"### Career Options for {major_name}\n\n"
        out += "**🤖 High AI (>0.5):**\n" + ("\n".join(f"- {t}" for t in high_ai) or "- None") + "\n\n"
        out += "**🛡️ Safe Haven (<0.2):**\n" + ("\n".join(f"- {t}" for t in safe) or "- None") + "\n\n"
        out += "**⚖️ Hybrid (0.2–0.5):**\n" + ("\n".join(f"- {t}" for t in hybrid) or "- None")

        return out

    except Exception as e:
        return f"Tool error in get_jobs_by_major: {type(e).__name__}: {e}"



def get_institutions_by_major(
    tool_context: ToolContext,
    major_name: Optional[str] = None,
    award_level: Optional[str] = None,
    modality: Optional[str] = None,
    state: Optional[str] = None,
    **kwargs
) -> str:
    if not bq_client:
        return "Error: BigQuery not initialized."

    # ---- Resolve major (from argument or state) ----
    if not major_name:
        major_name = tool_context.state.get("last_major_name")

    # ---- Collect missing inputs BEFORE returning (ask once) ----
    missing = []
    if not major_name:
        missing.append("**major/program** (e.g., Counseling, Clinical Mental Health Counseling, Psychology)")
    if not award_level:
        missing.append("**degree level** (Associate, Bachelor, Master, Doctorate, Undergraduate Certificate, Graduate Certificate)")
    if not modality:
        missing.append("**modality** (Online, Hybrid, On-campus)")

    if missing:
        return (
            "To recommend institutions, I need: "
            + ", ".join(missing)
            + ".\n\nOptional: add a **state** (e.g., CA, WA) to filter results."
        )

    # ---- Normalize modality (user-friendly -> edge codes) ----

    mod_kw = normalize_modality(modality)
    if not mod_kw:
        return "Modality not recognized. Try: **Online**, **Hybrid**, or **On-campus**."

    # 🔍 DEBUG: modality normalization
    print(f"🔍 Modality debug → user='{modality}' → normalized='{mod_kw}'")


    # ---- Normalize award level into internal bucket ----
    aw = normalize_award_level(award_level)

    # If user just says "certificate", ask them which one (or both)
    if aw == "certificate":
        return (
            "When you say **certificate**, do you mean:\n"
            "- **Undergraduate Certificate** (short-term / non-degree)\n"
            "- **Graduate Certificate** (post-bacc / post-master)\n"
            "- **Both**\n\n"
            "Reply with: **undergrad certificate**, **graduate certificate**, or **both**."
        )

    if not aw:
        return (
            "Degree level not recognized. Try: **Associate**, **Bachelor**, **Master**, **Doctorate**, "
            "**Undergraduate Certificate**, **Graduate Certificate**."
        )

    # ---- Build award-level filters (this is the “2)” part) ----
    award_filters = []
    if aw == "associate":
        award_filters.append("e.award_level = \"Associate's degree\"")
    elif aw == "bachelor":
        award_filters.append("e.award_level = \"Bachelor\"")
    elif aw == "master":
        award_filters.append("e.award_level = \"Master's degree\"")
    elif aw == "doctor":
        award_filters.extend([
            "e.award_level LIKE 'Doctor%'",     # Doctor's degree - ...
            "e.award_level LIKE 'Doctoral%'"    # Doctoral_Other
        ])
    elif aw == "ug_cert":
        award_filters.append("e.award_level LIKE 'Certificates of%'")
    elif aw == "grad_cert":
        award_filters.extend([
            "e.award_level = \"Post-master's certificate\"",
            "e.award_level = \"Postbaccalaureate certificate\""
        ])
    elif aw == "both_certs":
        award_filters.extend([
            "e.award_level LIKE 'Certificates of%'",              # undergrad certs
            "e.award_level = \"Post-master's certificate\"",
            "e.award_level = \"Postbaccalaureate certificate\""
        ])

    # ---- Query (multi-CIP, no LIMIT 1) ----

    major_pattern = "(?i)" + re.escape(major_name.strip())

    where = []
    params = [
        bigquery.ScalarQueryParameter("major", "STRING", major_pattern),
        bigquery.ScalarQueryParameter("mod", "STRING", mod_kw),
    ]

    # apply award filter
    if award_filters:
        where.append("(" + " OR ".join(award_filters) + ")")

    # modality is exact match in your edge
    where.append("e.modality = @mod")

    if state:
        where.append("LOWER(n.state) = LOWER(@st)")
        params.append(bigquery.ScalarQueryParameter("st", "STRING", state))

    sql = f"""
        WITH matching_cips AS (
          SELECT DISTINCT CAST(cip_code AS STRING) AS cip_code
          FROM `{GCP_PROJECT_ID}.{NODE_PROGRAMS}`
          WHERE REGEXP_CONTAINS(title, @major)
        )
        SELECT
          n.name AS institution_name,
          n.state,
          e.award_level,
          e.modality,
          SUM(e.total_completers) AS grads
        FROM `{GCP_PROJECT_ID}.{EDGE_INST_PROGRAM}` e
        JOIN `{GCP_PROJECT_ID}.{NODE_INSTITUTIONS}` n
          ON CAST(e.institution_id AS STRING) = CAST(n.institution_id AS STRING)
        WHERE CAST(e.cip_code AS STRING) IN (SELECT cip_code FROM matching_cips)
          AND {" AND ".join(where)}
        GROUP BY institution_name, state, award_level, modality
        HAVING grads > 0
        ORDER BY grads DESC
        LIMIT 10
    """

    cfg = bigquery.QueryJobConfig(query_parameters=params)
    df = bq_client.query(sql, job_config=cfg).to_dataframe()

    if df.empty:
        return (
            f"No institutions found for **{major_name}** with filters: "
            f"{award_level} / {modality}" + (f" / {state}" if state else "")
        )

    out = (
        f"### Top Schools for {major_name}\n"
        f"Filters: **{award_level}**, **{modality}**" + (f", **{state}**" if state else "") + "\n\n"
    )
    for i, r in df.iterrows():
        out += (
            f"{i+1}. **{r['institution_name']}** ({r['state']}) — "
            f"{r['award_level']} | {r['modality']} | {int(r['grads'])} grads\n"
        )
    return out





root_agent = LlmAgent(
    name="OBI_WAN_Navigator",
    model=Gemini(model="models/gemini-2.5-flash"),
    instruction="""
    You are OBI-WAN (Occupation-Based Index for Workforce AI Navigator).

    **MANDATORY OUTPUT RULES:**
    1. **For Skills:** The tool will provide a list of 15 items. You MUST display the full list. **DO NOT SUMMARIZE.** Display them as a clean bulleted list.
    2. **For Jobs:** Group them by "High AI" (>0.5) and "Safe Haven" (<0.2).
    3. **Institutions (critical):**
    - If the user asks for schools and any of these are missing: **major/program**, **degree level**, **modality**, you MUST ask for ALL missing fields in ONE question.
    - Use these user-friendly choices:
    - **Modality:** Online / Hybrid / On-campus
    - **Degree level:** Associate / Bachelor / Master / Doctorate / Undergraduate Certificate / Graduate Certificate
    - If user says only "certificate", you MUST ask: "Undergraduate Certificate, Graduate Certificate, or Both?"

    **TONE:** Concise, Data-Driven, Strategic. Stop giving generic advice; focus on the data lists.
    """,
    tools=[get_job_requirements, get_jobs_by_major, get_institutions_by_major]
)




#### V1. RAG without adapting Knowledge Graph

In [ ]:
%%writefile career_coach_agent/agent.py
import os
import json
from typing import Optional

import numpy as np
import pandas as pd

from kaggle_secrets import UserSecretsClient

# BigQuery / GCP
from google.cloud import bigquery
from google.oauth2 import service_account

# Gemini / Generative AI
import google.generativeai as genai

# ADK
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools.tool_context import ToolContext
from google.adk.runners import InMemoryRunner   # optional – keep if you use it
from google.adk.sessions import InMemorySessionService  # optional

# For ADK Web UI proxy in Kaggle
from IPython.core.display import display, HTML
from jupyter_server.serverapp import list_running_servers

# --- CONFIGURATION ---
GCP_PROJECT_ID = "dashboard-458701"
REGION = "us-west1"

# Table IDs
JOB_DATA_TABLE_ID = "onet.job_search_table"
JOB_EMBED_TABLE_ID = "onet.job_embeddings"
INST_DATA_TABLE_ID = "ipeds.institution_search_table"

# AI Score Data
AI_SCORE_TABLE_ID = "onet.final_ai_applicability_multiplicative"
AI_SCORE_COL = "AI_Applicability_Score"

# Embedding model constant
EMBED_MODEL = "models/text-embedding-004"

# --- INITIALIZATION ---
def initialize_services():
    """
    Initialize Gemini and BigQuery using Kaggle Secrets.
    Returns:
        bq_client: an authenticated BigQuery client.
    """
    try:
        user_secrets = UserSecretsClient()

        # 1. Setup Gemini
        google_api_key = user_secrets.get_secret("GOOGLE_API_KEY")
        os.environ["GOOGLE_API_KEY"] = google_api_key
        genai.configure(api_key=google_api_key)

        # 2. Setup BigQuery
        gcp_key_json = user_secrets.get_secret("GCP_KEY")
        credentials = service_account.Credentials.from_service_account_info(json.loads(gcp_key_json))
        bq_client = bigquery.Client(
            project=GCP_PROJECT_ID,
            credentials=credentials,
            location=REGION,
        )

        print("✅ BigQuery and Gemini initialized")
        return bq_client

    except Exception as e:
        print(f"❌ Initialization Error: {e}")
        return None

# Create global BigQuery client
bq_client = initialize_services()


# --- GLOBAL CACHES ---
job_registry_df = None
job_emb_matrix = None


# --- Registry Loader ---
def load_job_registry():
    global job_registry_df, job_emb_matrix
    if job_registry_df is None:
        print("⏳ Loading OBI-WAN Job Registry...")
        sql = f"""
            SELECT e.soc_code, e.Title, e.embedding, s.{AI_SCORE_COL} as ai_score
            FROM `{GCP_PROJECT_ID}.{JOB_EMBED_TABLE_ID}` e
            LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s
                ON e.soc_code = s.soc_code
            WHERE e.embedding IS NOT NULL
        """
        try:
            job_registry_df = bq_client.query(sql).to_dataframe()
            job_registry_df['ai_score'] = pd.to_numeric(job_registry_df['ai_score'], errors='coerce').fillna(0.5)
            job_emb_matrix = np.array(job_registry_df["embedding"].tolist())
            print(f"✅ OBI-WAN Registry Loaded: {len(job_registry_df)} jobs.")
        except Exception as e:
            print(f"❌ Error loading registry: {e}")


# --- HELPER FUNCTIONS ---
def get_ai_narrative(score):
    score = float(score)
    HIGH_THRESHOLD = 0.5
    LOW_THRESHOLD = 0.2

    if score >= HIGH_THRESHOLD:
        return (f"⚠️ **High AI Integration Zone (Score: {score:.2f})**")
    elif score <= LOW_THRESHOLD:
        return (f"🛡️ **Human-Centric Zone (Score: {score:.2f})**")
    else:
        return (f"⚖️ **Hybrid Zone (Score: {score:.2f})**")

def normalize_award_level(user_text: Optional[str]) -> Optional[str]:
    if not user_text: return None
    t = user_text.lower()
    if "bachelor" in t or "undergrad" in t: return "bachelor"
    if "master" in t or "graduate" in t and "doctor" not in t: return "master"
    if "post-master" in t or "post master" in t: return "post-master"
    if "postbacc" in t or "post-bacc" in t: return "postbacc"
    if "associate" in t or "assoc" in t: return "associate"
    if "certificate" in t or "cert" in t: return "certificate"
    return None

def normalize_modality(user_text: Optional[str]) -> Optional[str]:
    if not user_text: return None
    t = user_text.lower()
    if "online" in t or "remote" in t: return "DE ENTIRELY"
    if "hybrid" in t or "mixed" in t: return "DE SOME"
    if "campus" in t or "person" in t or "f2f" in t: return "F2F"
    return None


# --- TOOL 1: Get Job Requirements
def get_job_requirements(
    tool_context: ToolContext,
    job_title: str,
    category: str = "Skill"   # default so tool-calling works
) -> str:
    """
    Finds skills, knowledge, or abilities for a job title.
    Args:
        job_title: The job (e.g., "Translator").
        category: (optional) 'Skill', 'Knowledge', or 'Ability'.
                  Defaults to 'Skill' if not provided.
    """
    # 1. Make sure registry is loaded (embeddings + AI score)
    load_job_registry()
    if job_registry_df is None or job_registry_df.empty:
        return "Error: job registry is empty or could not be loaded."

    # 2. Semantic match job title → SOC code
    try:
        resp = genai.embed_content(model=EMBED_MODEL, content=job_title)
        q_vec = np.array(resp["embedding"])

        sims = np.dot(job_emb_matrix, q_vec) / (
            np.linalg.norm(job_emb_matrix, axis=1) * np.linalg.norm(q_vec) + 1e-8
        )
        best_idx = int(np.argmax(sims))
        best_row = job_registry_df.iloc[best_idx]

        best_title = best_row["Title"]
        best_soc = best_row["soc_code"]
        ai_score = best_row["ai_score"]
    except Exception as e:
        return f"Navigation Error: Could not locate job '{job_title}'. Details: {e}"

    # 3. Save best match for follow-ups
    tool_context.state["last_job_title"] = best_title

    # 4. Normalize category → DB values
    cat_lower = category.lower()
    if "knowledge" in cat_lower:
        db_category = "Knowledge"
    elif "ability" in cat_lower or "abilities" in cat_lower:
        db_category = "Ability"
    else:
        db_category = "Skill"

    # 5. Use parameterized query (old working pattern)
    job_config_items = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ScalarQueryParameter("category_param", "STRING", db_category),
            bigquery.ScalarQueryParameter("best_soc_code_param", "STRING", best_soc),
        ]
    )

    sql_get_items = f"""
        SELECT `Element Name` AS item_name
        FROM `{GCP_PROJECT_ID}.{JOB_DATA_TABLE_ID}`
        WHERE
            soc_code = @best_soc_code_param
            AND `Scale Name` = 'Importance'
            AND Category_ska = @category_param
        GROUP BY `Element Name`
        ORDER BY MAX(`Data Value`) DESC
        LIMIT 15
    """

    try:
        results = bq_client.query(sql_get_items, job_config=job_config_items).to_dataframe()
        if results.empty:
            skills_formatted = f"No specific {db_category} data found."
        else:
            # bullet list formatting
            skills_formatted = "\n".join(
                f"- {row['item_name']}" for _, row in results.iterrows()
            )
    except Exception as e:
        return f"Error retrieving skills from database: {e}"

    # 6. Combine with AI narrative
    return (
        f"### Analysis for {best_title}\n"
        f"{get_ai_narrative(ai_score)}\n\n"
        f"**Top {db_category}s:**\n{skills_formatted}"
    )


# TOOL 2: Get Jobs by Major
def get_jobs_by_major(tool_context: ToolContext, major_name: str) -> str:
    """
    Finds jobs for a college major and forecasts AI impact.
    Args:
        major_name: The major (e.g., "Psychology").
    """
    if not bq_client: return "Error: BigQuery not ready."
    tool_context.state['last_major_name'] = major_name

    sql = f"""
        SELECT DISTINCT j.Title, s.{AI_SCORE_COL} as score
        FROM `{GCP_PROJECT_ID}.{JOB_DATA_TABLE_ID}` j
        LEFT JOIN `{GCP_PROJECT_ID}.{AI_SCORE_TABLE_ID}` s ON j.soc_code = s.soc_code
        WHERE REGEXP_CONTAINS(j.CIPTitle, r'(?i){major_name}') AND j.Title IS NOT NULL LIMIT 50
    """
    try:
        df = bq_client.query(sql).to_dataframe()
        if df.empty: return f"No jobs found for major '{major_name}'."

        df['score'] = pd.to_numeric(df['score'], errors='coerce').fillna(0.5)

        HIGH = 0.5
        LOW = 0.2

        high_ai = df[df['score'] >= HIGH]['Title'].head(5).tolist()
        low_ai = df[df['score'] <= LOW]['Title'].head(5).tolist()
        hybrid_ai = df[(df['score'] > LOW) & (df['score'] < HIGH)]['Title'].head(5).tolist()

        resp = f"### Career Forecast for {major_name}\n"
        if high_ai: resp += f"**🤖 Augmentation Path (High AI):**\n" + ", ".join(high_ai) + "\n\n"
        if low_ai: resp += f"**🛡️ Safe Haven (Human-Centric):**\n" + ", ".join(low_ai) + "\n\n"
        if hybrid_ai: resp += f"**⚖️ Hybrid Roles:**\n" + ", ".join(hybrid_ai)
        return resp
    except Exception as e: return f"Error: {e}"



# TOOL 3: Get Institutions
def get_institutions_by_major(tool_context: ToolContext, major_name: Optional[str] = None, award_level: Optional[str] = None, modality: Optional[str] = None) -> str:
    """
    Finds top institutions for a major.
    Args:
        major_name: The major (e.g., "Psychology").
        award_level: "Bachelor", "Master", etc.
        modality: "Online", "On-Campus", etc.
    """
    if not bq_client: return "Error: BigQuery not initialized."

    if not major_name:
        major_name = tool_context.state.get('last_major_name')
        if not major_name: return "Please tell me which major you are interested in first."

    sql_base = f"FROM `{GCP_PROJECT_ID}.{INST_DATA_TABLE_ID}`"
    where_clauses = [f"REGEXP_CONTAINS(CIPTitle, r'(?i){major_name}')"]
    query_params = []

    award_keyword = normalize_award_level(award_level)
    if award_keyword:
        where_clauses.append("REGEXP_CONTAINS(LOWER(AWARD_LEVEL), @award_key)")
        query_params.append(bigquery.ScalarQueryParameter("award_key", "STRING", award_keyword))

    modality_code = normalize_modality(modality)
    if modality_code:
        where_clauses.append("MODALITY = @mod_code")
        query_params.append(bigquery.ScalarQueryParameter("mod_code", "STRING", modality_code))

    sql_query = f"""
        SELECT institution_name, SUM(CTOTALT) AS total_conferrals
        {sql_base}
        WHERE {" AND ".join(where_clauses)}
        GROUP BY institution_name
        HAVING total_conferrals > 0
        ORDER BY total_conferrals DESC LIMIT 10
    """

    try:
        job_config = bigquery.QueryJobConfig(query_parameters=query_params)
        results = bq_client.query(sql_query, job_config=job_config).to_dataframe()
        if results.empty: return f"No institutions found for '{major_name}' with those specific filters."

        inst_list = ", ".join([f"{i+1}. {row['institution_name']}" for i, row in results.iterrows()])
        return f"Top institutions for {major_name} ({award_level or 'All Levels'}, {modality or 'All Modalities'}): {inst_list}"
    except Exception as e: return f"Error: {e}"



# --- AGENT DEFINITION ---
root_agent = LlmAgent(
    name="OBI_WAN_Navigator",
    model=Gemini(model="models/gemini-2.5-flash"),

    instruction="""
    You are OBI-WAN (Occupation-Based Index for Workforce AI Navigator).

    **MANDATORY OUTPUT RULES:**
    1. **For Skills:** The tool will provide a list of 15 items. You MUST display the full list. **DO NOT SUMMARIZE.** Display them as a clean bulleted list.
    2. **For Jobs:** Group them by "High AI" (>0.5) and "Safe Haven" (<0.2).
    3. **For Institutions:** Ask for Degree Level and Modality before listing schools.

    **TONE:** Concise, Data-Driven, Strategic. Stop giving generic advice; focus on the data lists.
    """,
    tools=[get_job_requirements, get_jobs_by_major, get_institutions_by_major]
)


# 6. Deployment with ADK Web UI

In [ ]:
# Gets the proxied URL in the Kaggle Notebooks environment

from jupyter_server.serverapp import list_running_servers
from IPython.display import display, HTML

def get_adk_proxy_url():
    PROXY_HOST = "https://kkb-production.jupyter-proxy.kaggle.net"
    ADK_PORT = "8000"
    servers = list(list_running_servers())
    if not servers:
        raise Exception("No running Jupyter servers found.")
    baseURL = servers[0]["base_url"]
    try:
        path_parts = baseURL.split("/")
        kernel = path_parts[2]
        token = path_parts[3]
    except IndexError:
        raise Exception(f"Could not parse kernel/token from base URL: {baseURL}")
    url_prefix = f"/k/{kernel}/{token}/proxy/proxy/{ADK_PORT}"
    url = f"{PROXY_HOST}{url_prefix}"
    styled_html = f"""
    <div style="padding: 15px; border: 2px solid #f0ad4e; border-radius: 8px; background-color: #fef9f0; margin: 20px 0;">
        <div style="font-family: sans-serif; margin-bottom: 12px; color: #333; font-size: 1.1em;">
            <strong>⚠️ IMPORTANT: Action Required</strong>
        </div>
        <div style="font-family: sans-serif; margin-bottom: 15px; color: #333; line-height: 1.5;">
            The ADK web UI is <strong>not running yet</strong>. You must start it in the next cell.
            <ol style="margin-top: 10px; padding-left: 20px;">
                <li style="margin-bottom: 5px;"><strong>Run the next cell</strong> (the one with <code>!adk web ...</code>) to start the ADK web UI.</li>
                <li style="margin-bottom: 5px;">Wait for that cell to show it is "Running" (it will not "complete").</li>
                <li>Once it's running, <strong>return to this button</strong> and click it to open the UI.</li>
            </ol>
        </div>
        <a href='{url}' target='_blank' style="
            display: inline-block; background-color: #1a73e8; color: white; padding: 10px 20px;
            text-decoration: none; border-radius: 25px; font-family: sans-serif; font-weight: 500;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2); transition: all 0.2s ease;">
            Open ADK Web UI (after running cell below) ↗
        </a>
    </div>
    """
    display(HTML(styled_html))
    return url_prefix

# Get the URL prefix and display the button
url_prefix = get_adk_proxy_url()

# Run this cell after the one above
!adk web --log_level DEBUG --url_prefix {url_prefix}

/usr/local/lib/python3.11/dist-packages/google/adk/cli/fast_api.py:130: UserWarning: [EXPERIMENTAL] InMemoryCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  credential_service = InMemoryCredentialService()
/usr/local/lib/python3.11/dist-packages/google/adk/auth/credential_service/in_memory_credential_service.py:33: UserWarning: [EXPERIMENTAL] BaseCredentialService: This feature is experimental and may change or be removed in future versions without notice. It may introduce breaking changes at any time.
  super().__init__()
INFO:     Started server process [82]
INFO:     Waiting for application startup.

+-----------------------------------------------------------------------------+
| ADK Web Server started                                                      |
|                                                                             |
| For local testing, access at http:/

# 7. Technical Concepts Demonstrated

This project incorporates multiple core ADK concepts:

- **LLM-Powered Agent:** OBI-WAN uses Gemini 2.5 Flash via `LlmAgent` to power reasoning, skill mapping, and guided conversations.  
- **Custom Tools:** Three custom BigQuery tools retrieve real-time job skills, career forecasts, and institution information.  
- **Sequential Multi-Turn Workflow:** The agent maintains multi-turn state using ADK’s built-in session-based context passing, enabling a guided interview (degree level → modality → institutions).  
- **Agent Deployment:** The agent is deployed with `adk web`, providing an interactive UI for users inside the Kaggle environment.

# 8. How to Run / Reproduce

Step 1:  Configure API Keys

OBI-WAN requires two secrets stored in Kaggle Secrets:

`GCP_KEY` — Google Cloud service account JSON

`GOOGLE_API_KEY` — Gemini API key

These are loaded using:

In [ ]:
from kaggle_secrets import UserSecretsClient
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
GCP_KEY = UserSecretsClient().get_secret("GCP_KEY")

Step 2: Initialize Services

The project uses a single `initialize_services()` function to set up:

Google Gemini (`google.generativeai`)

BigQuery Client (for O*NET, IPEDS, embeddings, and AI applicability)

In [ ]:
bq_client = initialize_services()

Step 3: Load Required Tables in BigQuery

OBI-WAN depends on three tables:

`onet.job_search_table`

`onet.job_embeddings`

`ipeds.institution_search_table`

You must recreate them in your own BigQuery project using O*NET and IPEDS data.

Step 4: Register Tools

Three custom ADK tools power the agent:

`get_job_requirements`

`get_jobs_by_major`

`get_institutions_by_major`

These tools perform:

semantic search via embeddings

SQL retrieval

AI-risk contextualization

multi-turn state management

Step 5: Define the Agent

Once tools are defined, the OBI-WAN agent is registered via:

In [ ]:
root_agent = LlmAgent(
    model=Gemini("models/gemini-2.5-flash"),
    tools=[...]
)

Step 6: Run the Web UI (Optional)

In Kaggle Notebooks, the UI is viewable via:

In [ ]:
!adk web --log_level DEBUG --url_prefix {url_prefix}

The demonstration of the agent is shown in the accompanying YouTube video!

https://youtu.be/ULZau9wDRtA